---
title: "À quelques kilomètres près : lycées et géographie des cadres dans trois aires urbaines de province"
output-file: index.html
---

In [ ]:
# Le SSPCloud définit MPLBACKEND=Agg : sans cette ligne, les figures ne s'affichent pas
%matplotlib inline
import warnings

import pandas as pd
from IPython.display import display

from cadres_lycees import (
    analyse,
    cartes,
    collecte,
    config,
    figures,
    geo,
    h1,
    nettoyage,
    pipeline,
    tableaux,
)

# Sans BUCKET_S3, stockage.py avertit qu'il lit et écrit dans data/ : c'est le cas prévu d'un
# compte neuf (README). Les autres avertissements, dont le repli d'une copie gelée, restent
warnings.filterwarnings("ignore", message="Variables d'environnement manquantes")
# pandas coupe les cases à 50 caractères : les intervalles et les sha256 seraient tronqués
pd.set_option("display.max_colwidth", None)


def afficher_onglets(morceaux):
    """Imprime le balisage Quarto des onglets et affiche les cartes, dans l'ordre."""
    for morceau in morceaux:
        if isinstance(morceau, str):
            print(morceau)
        else:
            display(morceau)

## Résumé

Dans les aires urbaines de Nantes, Lyon et Strasbourg, les communes proches d'un lycée privé ou d'un lycée à forte valeur ajoutée comptent-elles plus de cadres que leurs voisines ? Notre base de 789 communes croise le recensement, l'annuaire de l'éducation, la valeur ajoutée des lycées et quatre autres infrastructures.

Entre communes contiguës, être plus près d'un lycée privé ne va pas de pair avec un écart net de part de cadres. À Lyon, pour 5 km de moins jusqu'au lycée privé, l'intervalle va de −1,0 à +2,7 points de cadres : la règle fixée à l'avance parle de « résultat nul établi », mais cet intervalle n'exclut pas un écart de quelques points, et il est sans doute trop étroit (4.5). À Nantes et à Strasbourg, les données ne permettent pas de conclure. La hausse des cadres entre 2011 et 2022 va davantage de pair avec le nombre de lycées proches qu'avec leur valeur ajoutée, selon un test qui désavantage la qualité (4.4).

Ce sont des associations : un lycée privé peut aussi s'implanter là où vivent déjà des cadres.

## 1. Introduction

### 1.1 Pourquoi les lycées

Choisir un logement, c'est souvent choisir une école. À Paris, Fack et Grenet (2010) trouvent qu'un écart-type de performance en plus pour le collège public de secteur va de pair avec des prix immobiliers plus élevés de 1,4 à 2,4 %, et que cet écart se réduit quand l'offre privée est proche : le privé sert d'option de repli face à la carte scolaire. Black (1999) compare des logements de part et d'autre d'une limite de secteur scolaire ; nos paires de communes contiguës en sont une version simplifiée.

Nous transposons la question aux lycées de province, dont la valeur ajoutée est publiée chaque année. Le poids du privé y varie fortement : environ 78 % des élèves du second degré sont scolarisés dans le public en métropole, contre six sur dix environ dans les académies de Nantes et de Rennes (DEPP, *Repères et références statistiques*). Dans l'Ouest, le privé est un choix courant, sans doute moins sélectif socialement. Comparer une aire de l'Ouest à deux autres permet de voir si le lien entre lycées et cadres en dépend.


### 1.2 Question et hypothèses

**Question.** À position égale dans l'aire, les communes qui offrent un meilleur accès à des lycées privés ou à forte valeur ajoutée comptent-elles plus de cadres que leurs voisines, et ce lien tient-il face aux autres infrastructures ? Les hypothèses ont été écrites avant de lire toute donnée sur les cadres.

| | Hypothèse | Comparaison principale | Statut |
|---|---|---|---|
| H1 | À distance égale du centre, plus de cadres là où l'accès aux lycées privés sous contrat est meilleur | Paires de communes contiguës (4.1), anneaux de distance (4.2) | verdict par aire |
| H2 | Ce lien est plus faible dans l'aire de l'Ouest (Nantes), où le privé est plus répandu | H1 aire par aire (4.3) | descriptive |
| H3 | La qualité des lycées accessibles, plus que leur nombre, accompagne la hausse de la part de cadres de 2011 à 2022 | Régression sur la qualité et le nombre (4.4) | verdict, 3 aires ensemble |
| H4 | Le lien est plus fort avec les lycées qu'avec les gares, les médecins et les équipements | Mêmes paires pour chaque infrastructure (5) | descriptive |

Les règles de verdict ont été fixées à l'avance et sont rappelées avec chaque résultat. H2 reste descriptive car une seule aire de l'Ouest ne permet pas de tester une différence ; H4, car aucune règle n'avait été fixée avant son estimation (annexe, « Décisions prises après les premiers résultats »).


### 1.3 Ce que le projet peut et ne peut pas montrer

Le projet décrit des associations. Quatre limites empêchent d'aller plus loin :

- **causalité inverse** : un lycée privé peut s'implanter ou se maintenir là où vivent déjà des familles aisées ;
- **sélection résidentielle** : les préférences des ménages ne sont pas observées ;
- **croissance démographique** : elle peut précéder l'ouverture de lycées et accompagne l'arrivée de cadres ; elle n'est pas contrôlée ;
- **mesure de l'accès** : distances à vol d'oiseau depuis le centroïde de la commune.

Un résultat nul bien établi est un résultat : nous disons alors quelle taille d'association l'intervalle de confiance exclut.

## 2. Construire la base : une ligne par commune, quatre familles de variables

### 2.1 Schéma du pipeline et tableau des sources

Le rapport reconstruit la base à chaque exécution, depuis des fichiers bruts dont l'empreinte `sha256` est vérifiée (annexe). Chaque étape est une fonction de `src/cadres_lycees/` qui échoue si une jointure perd des lignes.

```{mermaid}
flowchart LR
    A[Communes<br/>aires, contours] --> D[Accessibilité<br/>une fonction pour tous les points]
    C[Lycées<br/>annuaire, IPS, IVAL] --> D
    E[Gares, médecins,<br/>sport, culture] --> D
    B[Recensement<br/>cadres 2011 et 2022] --> G[Base communale]
    D --> G
    F[Contrôles<br/>distance au centre, densité, prix] --> G
    G --> P[Paires de communes contiguës]
```

| Source | Producteur | Accès | Usage |
|---|---|---|---|
| Recensement 2022 (évolution et structure de la population, séries historiques) | Insee | fichiers | cadres 2011 et 2022, densité |
| Annuaire de l'éducation | Éducation nationale | API, copie gelée | lycées, secteur, coordonnées |
| IPS et IVAL des lycées | DEPP | API, copies gelées | composition sociale, valeur ajoutée |
| Aires d'attraction des villes, table de passage | Insee | fichiers | périmètre, codes communes |
| Contours des communes et des IRIS | IGN, `cartiflette` | copies gelées | géométries, contiguïté |
| Gares, APL, Data ES, BPE 2025 | SNCF, DREES, Sports, Insee | API et fichiers, copies gelées | H4 |
| Statistiques DVF | DGFiP, Etalab | copie gelée | prix au m² (robustesse) |

: Sources de données du projet {#tbl-sources}

In [ ]:
# La base est reconstruite depuis les copies gelées ; final/ n'est qu'une sortie
base, lycees, paires = pipeline.construire_base(config.AIRES_RETENUES)

In [ ]:
#| label: tbl-effectifs
#| tbl-cap: "Effectifs aux étapes clés de la construction de la base, par aire."
tableaux.tableau_effectifs(analyse.effectifs_base(base, lycees, paires))

Chaque jointure est contrôlée : une commune ou un lycée perdu fait échouer le code. Les 153 lycées retenus des trois aires comptent parmi les 2 413 de métropole pris en compte pour l'accès. Le prix au m² manque presque partout dans l'aire de Strasbourg, l'Alsace-Moselle étant hors DVF.

### 2.2 Variable à expliquer

Nous mesurons la part des cadres et professions intellectuelles supérieures parmi les actifs de 25 à 54 ans (occupés et chômeurs ayant déjà travaillé), en 2011 et en 2022, et son évolution en points. Cette tranche d'âge est celle des parents de lycéens et écarte les retraités. La source donne les arrondissements de Lyon, avec la même définition aux deux dates. H1, H2 et H4 portent sur 2022, H3 sur l'évolution.


### 2.3 Variable d'intérêt : l'accessibilité aux lycées

Nous mesurons l'accès, et non la présence d'un lycée dans la commune. Sont retenus les lycées généraux, technologiques et polyvalents, publics et privés sous contrat. Depuis le centroïde de chaque commune, à vol d'oiseau :

- la distance au lycée privé le plus proche (mesure principale de H1) ;
- le nombre de lycées privés à 5 km (mesure secondaire) ;
- l'IVAL moyenne des lycées à 5 km, pondérée par leurs candidats au bac. L'IVAL (indicateur de valeur ajoutée des lycées) compare le taux d'accès de la seconde au bac observé à celui qu'on attendrait au vu du profil des élèves ; nous en prenons la moyenne des sessions 2012 à 2022.

Les 2 413 lycées retenus de métropole comptent, y compris hors des aires, pour que le bord de l'aire ne fausse pas les distances.

In [ ]:
#| label: tbl-acces
#| tbl-cap: "Accès des communes aux lycées et aux autres infrastructures, par aire."
tableaux.tableau_acces(analyse.resume_acces(base))

Le privé est bien moins accessible à Strasbourg : 4 % des communes y ont un lycée privé à 5 km, contre 16 % à Nantes et 23 % à Lyon, et la commune médiane en est à 16 km. Le lycée public est à 6 à 8 km en médiane dans les trois aires. À Strasbourg, le petit nombre de communes exposées limitera la précision.

### 2.4 Contrôles et mauvais contrôles

Les comparaisons se font à l'intérieur d'une aire, avec deux contrôles : la distance au centre de l'aire et le logarithme de la densité. La distance au lycée privé augmente avec la première et diminue avec la seconde ; sans ces contrôles, on retrouverait le gradient centre-périphérie (3.2).

Le niveau de vie, l'IPS des lycées (indice de position sociale, calculé à partir des professions des parents d'élèves) et la croissance de la population ne sont jamais des contrôles : ils dépendent en partie de la part de cadres. Le prix au m², lié à elle dans les deux sens, n'entre qu'en robustesse.


### 2.5 Encadré « choix et biais »

::: {.callout-note title="Choix et biais"}
- **Nomenclature.** Le recensement de 2022 code les professions en PCS 2020, celui de 2011 en PCS 2003. Selon l'Insee, les deux premiers niveaux, dont le groupe des cadres, sont inchangés ; les reclassements de professions détaillées ne sont pas vérifiés. Seule H3, qui compare 2011 et 2022, y est exposée.
- **Valeurs manquantes.** Aucune commune des trois aires n'est sans actif ni sans part de cadres. Ce qui manque est traité explicitement : IVAL absente pour 12 communes de la population de H3 (catégorie « sans IVAL »), prix au m² absent sous 10 ventes et dans toute l'Alsace-Moselle, 7 lycées hors contrat sans coordonnées, écartés de la robustesse qui ajoute le privé hors contrat. Toute autre donnée manquante fait échouer le code.
- **IVAL.** Mesure relative (elle situe un lycée par rapport à ce qu'on attend de ses élèves) et instable d'une session à l'autre, d'où la moyenne 2012-2022, qui couvre la période entre les deux recensements. Sa fiabilité au niveau des communes vaut 0,608 (@tbl-fiabilite) : au-dessus du seuil de 0,5 fixé à l'avance, H3 est testable.
- **Prix.** Les statistiques DVF couvrent juillet 2021 à juin 2026, après le recensement 2022 ; l'Alsace-Moselle en est absente. La robustesse au prix porte sur Lyon et Nantes seulement.
- **Lycées étrangers.** Absents de l'annuaire ; 55 communes de l'aire de Strasbourg sont à moins de 10 km de la frontière, et leur exclusion est testée.
- **Lycées agricoles.** Exclus partout, leur contrat avec le ministère de l'Agriculture n'étant pas couvert par nos règles ; la Loire-Atlantique en compte 15 dans le privé, une limite pour H2.
- **Vol d'oiseau depuis le centroïde.** Les distances depuis la mairie sont testées.
- **Lyon par arrondissements**, sauf dans la carte à curseur (3.2).
- **Petites communes.** Sous 200 actifs, la part de cadres, tirée d'un échantillon, est imprécise : aucune commune à Nantes, 33 à Lyon, 72 à Strasbourg (@tbl-effectifs). Leur exclusion est testée.
- **Pré-enregistrement imparfait.** Hypothèses, choix des aires et règles de verdict étaient écrits avant les résultats, mais les pentes de H1 et H4 ont été vues le 03/11. Le tag Git prévu pour figer le plan (`plan-fige`) n'a jamais été posé : la trace d'antériorité est la [pull request n° 2](https://github.com/gabrielwenger1-coder/DataScience/pull/2), dont le dernier commit ([`b782378`](https://github.com/gabrielwenger1-coder/DataScience/commit/b782378), plan du 21/10) est daté par GitHub. Quelques détails d'exécution ont été fixés le 03/11, après ces chiffres ; ils sont listés en annexe et datés dans le journal.
:::

## 3. Trois aires, trois profils

### 3.1 Pourquoi ces trois aires

Huit aires candidates ont été comparées sur six critères, sans aucune donnée sur les cadres (tableau de support ci-dessous, journal du 20/10). Paris est exclue pour le poids des transports en commun.

- **(a) Une aire de l'Ouest** : Nantes plutôt que Rennes : les deux ont une seule cellule d'anneaux exploitable (voir 4.2), et Nantes a plus de paires de voisines dont une seule a un lycée privé à 10 km (76 contre 71).
- **(b) Un contraste du privé** : 47 % des lycées à Nantes, 45 % à Lyon, 27 % à Strasbourg ; écart d'IPS entre privé et public de 11,8, 23,7 et 22,8 points.
- **(c) Des tailles comparables** : 1,04 million d'habitants à Nantes, 0,87 million à Strasbourg ; Lyon (2,33 millions) est la plus grande des huit ; elle est retenue malgré tout pour le critère (e), qui prime : c'est la seule aire où les anneaux sont comparables à 5 km, donc la seule où un résultat nul peut être établi (4.2).
- **(d) Une dizaine de lycées privés** : 18, 37 et 9 ; Strasbourg est juste sous le seuil.
- **(e) Un support suffisant** : Lyon est la seule aire où la comparaison par anneaux (4.2) est possible à 5 km ; à 10 km, elle l'est à Lyon, Nantes, Rennes et Strasbourg.
- **(f) La faisabilité** : la source donne les arrondissements de Lyon.

Strasbourg a été préférée à Bordeaux pour son contraste avec Nantes et ses anneaux comparables à 10 km. Sa fragilité était connue : 11 communes exposées à 5 km.

In [ ]:
#| column: page
#| label: fig-aires
#| fig-cap: >-
#|   Part du privé sous contrat parmi les lycées et les candidats au bac, et IPS de chaque
#|   lycée par secteur, dans les 8 aires candidates (aires retenues en gras).
# La `fig_aires` montre les 8 candidates ; la base ne porte que sur les 3 aires retenues
referentiel_candidates = geo.referentiel_communes(config.AIRES_CANDIDATES)
lycees_8_aires = pipeline.lycees_avec_aire(referentiel_candidates)
profil = analyse.profil_aires(lycees_8_aires)
figures.fig_aires(
    profil,
    lycees_8_aires,
    "À Nantes, le privé est répandu et peu sélectif ; à Strasbourg, rare et sélectif",
    config.AIRES_RETENUES,
)

Le privé est aussi répandu à Nantes qu'à Lyon (47 % et 45 % des lycées), mais l'écart d'IPS entre privé et public y est deux fois plus faible (+12 contre +24 points). À Strasbourg, le privé est rare (16 % des candidats au bac) et aussi sélectif qu'à Lyon. La prémisse de H2 se vérifie : dans l'Ouest, le privé est plus répandu et moins sélectif.

In [ ]:
#| output: asis
# Replié : le tableau de support justifie le choix sans alourdir la lecture
criteres_aires, support_aires, _ = analyse.lire_tableau_support()
print(
    tableaux.encadre_replie(
        "Tableau de support du choix des aires",
        tableaux.tableau_support(criteres_aires, support_aires),
    )
)

### 3.2 Le gradient centre-périphérie des cadres

La part de cadres baisse avec la distance au centre. Comparer des communes proches et éloignées d'un lycée privé sans tenir compte de leur position reviendrait donc à comparer le centre et la périphérie.

In [ ]:
#| label: tbl-gradient
#| tbl-cap: "Part de cadres des 25-54 ans en 2022 par anneau de distance au centre."
# Le gradient centre-périphérie justifie de comparer les communes à distance égale du centre
tableaux.tableau_gradient(analyse.parts_par_anneau(base))

La part de cadres est 1,9 à 2,5 fois plus élevée à moins de 5 km du centre qu'au-delà de 30 km : 39 % contre 17 % à Lyon, 36 % contre 15 % à Nantes, 29 % contre 16 % à Strasbourg. À Lyon, l'anneau de 5 à 10 km (23 %) passe sous celui de 10 à 20 km (26 %). À Nantes et à Strasbourg, l'anneau 0-5 km ne compte que 2 communes : sa valeur est celle de la commune-centre et d'une voisine.

In [ ]:
#| output: asis
# Chaque recensement de la série historique est ramené au COG 2025
series_csp = {annee: collecte.serie_csp_2554(annee) for annee in config.GEOGRAPHIE_SERIE_CSP}
parts_dates = nettoyage.parts_cadres_dates(series_csp, collecte.passage_communes(), base)
cartes_dates = {
    aire: cartes.carte_cadres_dates(base[base["aire"] == aire], parts_dates, f"Aire de {aire}")
    for aire in config.AIRES_RETENUES
}
legende = (
    "**La part de cadres baisse du centre vers la périphérie à chaque recensement "
    "depuis 1999.** Part des cadres parmi les actifs de 25 à 54 ans par commune, de"
    " 1999 à 2022, une aire par onglet et un curseur pour l'année."
)
afficher_onglets(cartes.onglets_par_aire("fig-cadres-dates", cartes_dates, legende))

Ce gradient se retrouve à chaque recensement depuis 1999, tandis que la part de cadres monte dans les trois aires. La carte ne montre pas les lycées et ne dit rien de H1.

### 3.3 La géographie de l'accès aux lycées

In [ ]:
#| output: asis
cartes_acces = {
    aire: cartes.carte_acces(base[base["aire"] == aire], lycees, f"Aire de {aire}")
    for aire in config.AIRES_RETENUES
}
legende = (
    "**Hors des villes-centres, le lycée privé le plus proche est souvent à plus de"
    " 10 km.** Distance à vol d'oiseau du centroïde de chaque commune au lycée "
    "privé sous contrat le plus proche, lycées publics et privés en points, une "
    "aire par onglet."
)
afficher_onglets(cartes.onglets_par_aire("fig-acces", cartes_acces, legende))

Les lycées privés sont regroupés dans les villes-centres et quelques pôles secondaires. Au-delà, la distance augmente vite, surtout à Strasbourg, où la plupart des communes sont à plus de 10 km d'un lycée privé. Les communes exposées sont donc plus centrales que les autres : la section 4 compare à position égale.

## 4. À position égale, un meilleur accès aux lycées va-t-il de pair avec plus de cadres ?

### 4.1 Comparaison entre communes contiguës

Le résultat principal compare des **paires contiguës**, deux communes de la même aire qui ont une frontière ou un point commun : 301 paires à Nantes, 1 127 à Lyon, 717 à Strasbourg. Deux voisines partagent marché de l'emploi et position dans l'aire, qui s'annulent dans la différence. On régresse l'écart de part de cadres sur l'écart de distance au lycée privé le plus proche, avec les écarts de distance au centre et de densité. H1 attend une pente négative.

Une commune entre dans plusieurs paires : les intervalles viennent d'un **bootstrap par commune**, qui tire les communes et non les paires (1 000 tirages).

In [ ]:
#| column: page
#| label: fig-paires
#| fig-cap: >-
#|   Écart de part de cadres en 2022 et écart de distance au lycée privé le plus proche entre
#|   communes contiguës, à position égale, moyennes par classes et pente principale.
pentes_principales = h1.pentes_paires_par_aire(paires, "d_dist_lyc_prive_min")
figures.fig_paires(
    paires,
    pentes_principales,
    "Entre communes voisines, la pente est faible et son intervalle contient zéro partout",
)

La pente est négative dans les trois aires, mais faible : −0,15 point de cadres par km à Nantes, −0,18 à Lyon, −0,06 à Strasbourg, et les trois intervalles contiennent zéro. À Lyon, l'intervalle [−0,53 ; +0,21] signifie qu'être 5 km plus près d'un lycée privé s'accompagne d'un écart compris entre −1,0 et +2,7 points de cadres. Ce n'est pas négligeable : à Lyon, la part de cadres perd 22 points du centre à l'au-delà de 30 km (@tbl-gradient), de l'ordre d'un demi-point par km, et la borne basse de la pente, −0,53 point par km de distance au lycée privé, est du même ordre. Le « résultat nul établi » de Lyon (4.2) dit que les données ne distinguent pas l'association de zéro, pas qu'elle est petite.

Le nuage est symétrique autour de zéro par construction : chaque paire y figure dans les deux sens, puisque l'ordre des deux communes est arbitraire.

### 4.2 Comparaison par anneaux et verdict de H1

La figure ci-dessous rassemble les estimations qui entrent dans le verdict de H1 : les pentes principale et secondaire des paires (4.1), puis les anneaux.

Dans chaque anneau de distance au centre (0-5, 5-10, 10-20, 20-30, plus de 30 km), on compare les communes qui ont un lycée privé dans le rayon (exposées) aux autres, avec les contrôles de 2.4. Une cellule (aire × anneau) n'est estimée que si chaque groupe compte au moins 15 communes et si leurs distances moyennes au centre diffèrent de 2 km au plus ; sinon, on fusionne des anneaux voisins. Les intervalles viennent d'un bootstrap : 1 000 tirages des communes avec remise. Seules 6 cellules passent ces règles : 2 à Lyon à 5 km ; à 10 km, 2 à Lyon, 1 à Nantes, 1 à Strasbourg (panneau de droite).

In [ ]:
#| column: page
#| label: fig-h1
#| fig-cap: >-
#|   Estimations de H1 par aire, avec intervalles à 95 % : pentes principale et secondaire des
#|   paires contiguës, écarts des cellules d'anneaux estimables à 5 et 10 km.
# Les deux pentes voient les mêmes tirages du bootstrap ; la `fig_h1` sert aussi H2
pentes_secondaires = h1.pentes_paires_par_aire(paires, "d_n_lyc_prive_5km")
anneaux = h1.ecarts_anneaux(base)
figures.fig_h1(
    pentes_principales,
    pentes_secondaires,
    anneaux,
    "Aucune pente des paires n'exclut zéro ; une seule cellule d'anneau l'exclut",
)

Les quatre cellules de Lyon vont dans le sens de H1 (+1,5 à +3,1 points de cadres dans les communes exposées), mais aucun intervalle n'exclut zéro ; elles concordent entre les deux rayons. Seule la cellule de Nantes au-delà de 20 km exclut zéro : +2,0 points [+0,1 ; +3,9]. Les anneaux ne tranchent pas seuls ; ils servent au verdict des paires.

Verdicts, selon les règles fixées à l'avance :

- **Lyon : « infirmée » (résultat nul établi).** L'intervalle contient zéro, les deux anneaux comparés aux rayons de 5 et 10 km concordent, et aucune cellule n'exclut zéro. Toutes les estimations de Lyon vont pourtant dans le sens de H1 (pente négative, pente secondaire et cellules positives), sans qu'aucune n'exclue zéro : « infirmée » est le nom, fixé à l'avance, d'un lien trop faible pour être distingué de zéro, pas d'un lien de sens contraire.
- **Nantes et Strasbourg : « non concluante ».** L'intervalle contient zéro, mais sans anneau estimable aux deux rayons, la règle interdit de parler de résultat nul établi.

La pente secondaire, sur le nombre de lycées privés à 5 km, a un intervalle qui contient zéro partout (@fig-h1). Elle va dans le sens de H1 à Lyon (+0,41 point de cadres par lycée), mais dans le sens contraire à Nantes (−0,46) et à Strasbourg (−0,65) : dans ces deux aires, elle aurait de toute façon empêché une confirmation, et le verdict reste « non concluante ».

### 4.3 L'Ouest est-il différent ?

Les pentes principales des trois aires sont très proches (@fig-h1) et leurs intervalles se recouvrent largement : rien ne distingue Nantes. La seule estimation qui exclut zéro, la cellule nantaise au-delà de 20 km, irait même à l'inverse de H2, mais elle est isolée. Un lien faible partout laisse peu de place à des écarts entre aires ; H2 reste descriptive.